# Fase 4 - Sensibilidad de umbral y clasificación con umbral alternativo

## Objetivo
Evaluar el efecto de distintos umbrales de esfuerzo sobre la clase "accesible". Con el umbral 0.30 no hay clase 1, por lo que se busca un umbral alternativo que permita tener ambas clases y analizar falsos positivos / falsos negativos con datos reales.

## Insumos
- `data/integrado/maestro_joven_uk.parquet`
- `data/limpia/hipotecas_limpio.parquet`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from proyecto import config
from proyecto import indicadores

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import confusion_matrix, classification_report

CARPETA_FIGURAS = config.MD_FIGURAS

maestro = pd.read_parquet(config.RUTA_MAESTRO).copy()
hipotecas = pd.read_parquet(config.RUTA_HIPOTECAS_LIMPIO).copy()

tasa = float(hipotecas['tasa_inicial_pct'].mean())
comisiones = float(hipotecas['comisiones_total'].mean())

df = maestro.copy()
df['cuota_mensual'] = df.apply(
    lambda r: indicadores.cuota_mensual_pmt(r['precio_mediana'], tasa, config.PLAZO_ANIOS, config.LTV, comisiones),
    axis=1,
)
df['salario_joven_mensual'] = df['salario_joven_region'] / 12.0
df['esfuerzo'] = df['cuota_mensual'] / df['salario_joven_mensual']

df['sol_anom'] = df['sol_anom'].fillna(0.0)
df['precio_var_pct'] = df['precio_var_pct'].fillna(0.0)

## 1. Distribución del esfuerzo y sensibilidad por umbral

In [ ]:
print("=== 1. DISTRIBUCIÓN DEL ESFUERZO ===")
print(df['esfuerzo'].describe())

umbrales = [0.30, 0.35, 0.40, 0.45, 0.50]
resumen = []
for u in umbrales:
    tmp = df.copy()
    tmp['accesible_u'] = (tmp['esfuerzo'] <= u).astype(int)
    cnt = tmp['accesible_u'].value_counts().to_dict()
    resumen.append({
        "umbral": u,
        "accesible(1)": int(cnt.get(1, 0)),
        "no_accesible(0)": int(cnt.get(0, 0)),
        "pct_accesible": round(tmp['accesible_u'].mean() * 100, 2)
    })

res_um = pd.DataFrame(resumen)
print("\n=== SENSIBILIDAD POR UMBRAL ===")
print(res_um.to_string(index=False))

## Figura 1 - Histograma con umbrales

In [ ]:
CARPETA_FIGURAS.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4.6))
datos = df['esfuerzo'].dropna()
ax.hist(datos, bins=20, color='#0f766e', edgecolor='white')
ax.axvline(0.30, color='#dc2626', ls='--', lw=2, label="Umbral 0.30")
ax.axvline(0.40, color='#f59e0b', ls='--', lw=2, label="Umbral 0.40")
ax.axvline(0.50, color='#2563eb', ls='--', lw=2, label="Umbral 0.50")
ax.axvline(datos.median(), color='#22c55e', ls=':', lw=2, label=f"Mediana {datos.median():.3f}")
ax.set_xlabel("Esfuerzo (cuota/salario joven mensual)")
ax.set_ylabel("Frecuencia")
ax.set_title("Sensibilidad de umbral sobre la distribución del esfuerzo")
ax.legend()
plt.tight_layout()
plt.savefig(CARPETA_FIGURAS / "fase4_hist_esfuerzo_umbral.png", dpi=110)
plt.show()

## Figura 2 - Barras de proporción accesible por umbral

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar([str(x) for x in res_um['umbral']], res_um['pct_accesible'], color='#1f77b4', edgecolor='white')
ax.set_xlabel("Umbral esfuerzo <= x")
ax.set_ylabel("% accesible")
ax.set_title("Proporción accesible según umbral")
plt.tight_layout()
plt.savefig(CARPETA_FIGURAS / "fase4_barras_umbral.png", dpi=110)
plt.show()

## 2. Clasificación con umbral alternativo (0.50)
Con el umbral 0.50 aparecen ambas clases (16 accesibles / 44 no accesibles), lo que permite analizar falsos positivos / falsos negativos.

In [ ]:
umbral_sel = 0.50

print("\n=== 2. CLASIFICACIÓN CON UMBRAL ALTERNATIVO ===")
print(f"Umbral seleccionado: esfuerzo <= {umbral_sel}")
df['accesible_sel'] = (df['esfuerzo'] <= umbral_sel).astype(int)
print(df['accesible_sel'].value_counts())
print(f"Proporción 1: {df['accesible_sel'].mean():.3f}")

print("\nPor región (umbral 0.50):")
print(df.groupby('region')['accesible_sel'].sum().sort_values(ascending=False).to_string())

## Árbol de decisión con umbral 0.50

In [ ]:
num_vars = ["precio_mediana", "salario_joven_region", "anos_salario", "precio_var_pct",
            "tmax_anom", "tmin_anom", "lluvia_anom", "sol_anom", "af_anom"]
X = df[num_vars + ["region"]].copy()
X = pd.get_dummies(X, columns=["region"], drop_first=True)
y = df['accesible_sel']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}")

print("\nPRUEBA DE PROFUNDIDAD")
print(f"{'max_depth':>10} {'acc_train':>12} {'acc_test':>10}")
for prof in [2, 3, 4, 5]:
    arbol = DecisionTreeClassifier(max_depth=prof, random_state=42)
    arbol.fit(X_train, y_train)
    print(f"{prof:>10} {arbol.score(X_train,y_train):>12.3f} {arbol.score(X_test,y_test):>10.3f}")

In [ ]:
print("\n=== ÁRBOL FINAL (max_depth=3) ===")
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_train, y_train)
print(f"Train: {arbol.score(X_train,y_train):.3f} | Test: {arbol.score(X_test,y_test):.3f}")
print("\nREGLAS:")
try:
    print(export_text(arbol, feature_names=list(X.columns), max_depth=3))
except Exception as e:
    print(e)

## Matriz de confusión y falsos positivos / falsos negativos

In [ ]:
y_pred = arbol.predict(X_test)

print("\n=== MATRIZ DE CONFUSIÓN (test) - labels [0,1] ===")
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(cm)

if cm.shape == (2, 2):
    print(f"VN(0,0):{cm[0,0]} FP(0,1):{cm[0,1]} FN(1,0):{cm[1,0]} VP(1,1):{cm[1,1]}")
    print("\n=== FOCO FALSOS POSITIVOS / FALSOS NEGATIVOS ===")
    print(f"Falsos positivos: {cm[0,1]}")
    print(f"Falsos negativos: {cm[1,0]}")

print("\n=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=["no_accesible(0)", "accesible(1)"], zero_division=0))

print("\n=== IMPORTANCIA TOP 10 ===")
imp = pd.Series(arbol.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(10).round(4).to_string())

## Conclusión Fase 4
- El umbral 0.30 es muy estricto: ninguna región es accesible.
- Con umbral 0.50 aparecen ambas clases (16 accesibles / 44 no accesibles).
- Con `max_depth=3` el árbol clasifica bien y permite medir falsos positivos/negativos con datos reales.
- Figuras generadas: `md/figuras/fase4_hist_esfuerzo_umbral.png`, `md/figuras/fase4_barras_umbral.png`